In [ ]:
with max_status_reached as (
select
   psl.subscription_id,
   max(psl.status_id) as max_status
from
   public.payment_status_log psl   

group by
   1   
),
payment_funnel_stages as (
select
   subs.subscription_id,
   date_trunc('year', order_date) as order_year,
   current_payment_status,
   max_status,
   case 
      when max_status = 1 then 'Payment Widget Opened'
      when max_status = 2 then 'Payment Entere'
      when max_status = 3 and current_payment_status != 0 then 'Payment Submitted'
      when max_status = 3 and current_payment_status = 0 then 'User Error With Payment Submission'
      when max_status = 4 and current_payment_status != 0 then 'Payment Success With Vendor'
      when max_status = 4 and current_payment_status = 0 then 'Payment Processing Error With Vendor'
      when max_status = 5 then 'Complete'
      when max_status is null then 'User Has Not Started Payment process'
      end as payment_funnel_stage
from
   public.subscriptions subs
left join
   max_status_reached m
   on subs.subscription_id = m.subscription_id      
)
select
   payment_funnel_stage,
   order_year,
   count(*) as num_subs

from
   payment_funnel_stages  
group by
   1, 2    
order by
   2 desc

In [6]:
with error_subs as(
select
   distinct subscription_id
from
   public.payment_status_log
where
   status_id = 0
)  
select
   subs.subscription_id,
   case
      when err.subscription_id is not null then 1
      else 0
      end as has_error
from
   public.subscriptions subs
left join
   error_subs err
   on subs.subscription_id = err.subscription_id 